# Part 8 — Paper 1: re-running the old real-data tables from raw files
Several Paper 1 tables came from scripts of the earlier version that no longer exist, so the paper labels them "earlier scripts". This notebook rebuilds them from Binance's public 1-second spot klines, so every real-data number in Paper 1 becomes reproducible. It also adds:
- a **placebo pipeline** for every pair;
- a **second fee event**: BTC zero fees started on 2022-07-08 and ended on 2023-03-22, so the fee test now has an "on" and an "off" event.

No pre-registration is needed: these are re-runs of existing designs, not new hypotheses.

**Run all cells in order.** Expected time: about 1–2 hours, for about 1,400 daily 1-second files.

In [ ]:
# ============================================================================
# CELL V0 — SETUP (self-contained; run first)
# Part 8 (Paper 1): re-runs, from raw public files, every real-data table that the earlier version produced
# with scripts we no longer have: the 96-date x 8-pair spot panel (three timing constructions), asset means
# with 30 flow lags, a placebo pipeline per pair, and the fee-schedule test, now with TWO fee events
# (BTC zero-fee introduced 2022-07-08; reinstated 2023-03-22).
# ============================================================================
import os, sys, json, time, math, glob, re, io, zipfile, hashlib, subprocess, warnings, zlib
import urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed
import xml.etree.ElementTree as ET

IN_COLAB = "google.colab" in sys.modules
try:
    import pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow"], check=True)
import numpy as np
import pandas as pd
from scipy import stats, optimize
warnings.filterwarnings("ignore", category=RuntimeWarning)

USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/ceiling_suite"
elif IN_COLAB:
    BASE = "/content/ceiling_suite"
else:
    BASE = os.path.abspath("ceiling_suite_local")
RESULTS, PREREG = os.path.join(BASE, "results"), os.path.join(BASE, "prereg")
XDATA = os.path.join(BASE, "data_xsec")
for _d in (RESULTS, PREREG, XDATA):
    os.makedirs(_d, exist_ok=True)

SEED = 20261023
def rng(offset=0):
    return np.random.default_rng(SEED + offset)

def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, np.ndarray):
        return _clean(o.tolist())
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return o

def record(test_id, passed, kind, metrics, criterion="", note=""):
    assert kind in {"theorem-check", "simulation", "real-data", "audit"}, kind
    status = "REPORT" if passed is None else ("PASS" if passed else "FAIL")
    out = {"test_id": test_id, "status": status, "kind": kind, "criterion": criterion, "note": note,
           "metrics": _clean(metrics), "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
           "seed": SEED}
    with open(os.path.join(RESULTS, f"{test_id}.json"), "w") as f:
        json.dump(out, f, indent=2)
    print(f"[{status}] {test_id} ({kind})" + (f" — {criterion}" if criterion else ""))
    if note:
        print("        note:", note)
    return out

def boot_ci(v, R, B=2000, stat=np.mean):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if len(v) < 3:
        return [np.nan, np.nan]
    bs = [stat(v[R.integers(0, len(v), len(v))]) for _ in range(B)]
    return [float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))]

# ---- Design (fixed; same as the earlier version where it existed) -------------------------------
SPOT = "https://data.binance.vision/data/spot/daily/klines"
PAIRS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "ADAUSDT", "DOGEUSDT", "BNBUSDT", "XRPUSDT", "LTCUSDT"]
PANEL_D = [d.strftime("%Y-%m-%d") for d in pd.date_range("2021-01-01", "2024-12-31", freq="D") if d.day in (10, 24)]
BAR, EMA_W = 10, 300                 # 10-second bars, 300-second EMA  ->  alpha = 2/31, mechanical persistence 29/31
FEE_EVENTS = {"zero_fee_on": "2022-07-08", "zero_fee_off": "2023-03-22"}
FEE_WIN = 20                         # days before and after each event, every day
THREADS = 8
SPOT_DIR = os.path.join(BASE, "data_spot"); os.makedirs(SPOT_DIR, exist_ok=True)
print("BASE =", BASE, "| panel dates:", len(PANEL_D))
def _http(url, method="GET", timeout=60, tries=3):
    last = None
    for i in range(tries):
        try:
            req = urllib.request.Request(url, method=method, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.status, (r.read() if method == "GET" else b"")
        except urllib.error.HTTPError as e:
            if e.code in (403, 404):
                return e.code, b""
            last = e
        except Exception as e:
            last = e
        time.sleep(1.5 * (i + 1))
    raise RuntimeError(f"network error {url}: {last}")
DL_LOG = []
def fetch_1s(sym, date):
    url = f"{SPOT}/{sym}/1s/{sym}-1s-{date}.zip"
    code, raw = _http(url, timeout=240)
    if code != 200:
        return None
    sha = hashlib.sha256(raw).hexdigest(); c2, chk = _http(url + ".CHECKSUM", timeout=30)
    if c2 == 200 and chk and chk.decode().split()[0] != sha:
        raise RuntimeError(f"SHA-256 mismatch {url}")
    DL_LOG.append({"symbol": sym, "date": date, "sha256": sha, "checksum_verified": bool(c2 == 200)})
    return raw


In [ ]:
# ============================================================================
# CELL V1 — FUNCTIONS
# ============================================================================
KCOLS = ["open_time", "open", "high", "low", "close", "volume", "close_time", "quote_volume", "count",
         "taker_buy_volume", "taker_buy_quote_volume", "ignore"]
def bars10(sym, date):
    """10-second bars from 1-second klines: last close (filled), signed taker quote flow, quote volume."""
    raw = fetch_1s(sym, date)
    if raw is None:
        return None
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        txt = z.read([n for n in z.namelist() if n.endswith(".csv")][0])
    k = pd.read_csv(io.BytesIO(txt), header=None, names=KCOLS)
    if not str(k.iloc[0, 0]).replace(".", "").isdigit():
        k = k.iloc[1:]
    t = pd.to_numeric(k["open_time"]).to_numpy(); t = np.where(t > 1e14, t // 1000, t)       # microseconds -> ms
    t0 = pd.Timestamp(date, tz="UTC").value // 10 ** 6
    b = ((t - t0) // (1000 * BAR)).astype(int); nb = 86400 // BAR; ok = (b >= 0) & (b < nb)
    qv = pd.to_numeric(k["quote_volume"]).to_numpy(); tb = pd.to_numeric(k["taker_buy_quote_volume"]).to_numpy()
    c = pd.to_numeric(k["close"]).to_numpy()
    D = pd.DataFrame({"b": b[ok], "c": c[ok], "f": (2 * tb - qv)[ok], "qv": qv[ok]})
    g = D.groupby("b"); idx = np.arange(nb)
    out = pd.DataFrame({"close": g["c"].last().reindex(idx).ffill().bfill(), "flow": g["f"].sum().reindex(idx).fillna(0.0),
                        "qv": g["qv"].sum().reindex(idx).fillna(0.0)})
    return out, float(g.size().reindex(idx).notna().mean())

def var_indices(X, F, extra=None):
    """VAR(1) of z_t = (X_t, F_t) on z_{t-1} (+ optional exogenous lags): Q, S, L and J."""
    Y = np.column_stack([X[1:], F[1:]]); R = [np.ones(len(Y)), X[:-1], F[:-1]]
    if extra is not None:
        R += [e[:-1] for e in extra]
    Z = np.column_stack(R); ok = np.all(np.isfinite(Y), 1) & np.all(np.isfinite(Z), 1)
    if ok.sum() < 500:
        return None
    B = np.linalg.lstsq(Z[ok], Y[ok], rcond=None)[0]; J = B[1:3].T
    if abs(1 - J[1, 1]) < 1e-9:
        return None
    Q = J[0, 1] * J[1, 0] / (1 - J[1, 1])
    return {"Q": float(Q), "S": float(J[0, 0] + Q), "J11": float(J[0, 0]), "J22": float(J[1, 1]),
            "L": float(Q / (1 - J[0, 0])) if abs(1 - J[0, 0]) > 1e-9 else np.nan}

def unit_stats(sym, date, seed=0):
    r = bars10(sym, date)
    if r is None:
        return None
    B, cover = r
    p = np.log(B["close"].to_numpy()); f = B["flow"].to_numpy() / 1e6
    al = 2 / (EMA_W / BAR + 1); m = pd.Series(p).ewm(alpha=al, adjust=False).mean().to_numpy(); x = p - m
    ret = np.r_[np.nan, np.diff(p)]
    f_next = np.r_[f[1:], np.nan]
    lags30 = [np.r_[np.full(j, np.nan), f[:-j]] for j in range(1, 31)]
    rr = np.random.default_rng(zlib.crc32(f"{sym}{date}{seed}".encode()))
    pl = np.cumsum(rr.normal(0, np.nanstd(np.diff(p)), len(p))); ml = pd.Series(pl).ewm(alpha=al, adjust=False).mean().to_numpy()
    out = {"symbol": sym, "date": date, "coverage": cover, "quote_volume_musd": float(B["qv"].sum() / 1e6), "n_bars": len(p)}
    for name, (X, F, ex) in {"forward": (x, f_next, None), "observed": (x, f, None), "raw_return": (ret, f, None),
                             "observed_q30": (x, f, lags30), "placebo": (pl - ml, f, None)}.items():
        v = var_indices(X, F, ex)
        for kk, vv in (v or {}).items():
            out[f"{name}_{kk}"] = vv
    return out

def date_mean_ci(U, col):
    """Equal-weight asset means within date, then the mean over dates with a Student t interval."""
    d = U.groupby("date")[col].mean().dropna(); n = len(d)
    if n < 3:
        return {"mean": np.nan, "ci": [np.nan, np.nan], "n_dates": n}
    se = d.std(ddof=1) / np.sqrt(n); t = stats.t.ppf(0.975, n - 1)
    return {"mean": float(d.mean()), "ci": [float(d.mean() - t * se), float(d.mean() + t * se)], "n_dates": n,
            "share_units_pos": float((U[col] > 0).mean())}
UNIT_DIR = os.path.join(SPOT_DIR, "units"); os.makedirs(UNIT_DIR, exist_ok=True)
def get_unit(sym, date):
    fp = os.path.join(UNIT_DIR, f"{sym}_{date}.json")
    if os.path.exists(fp):
        return json.load(open(fp))
    u = unit_stats(sym, date)
    json.dump(_clean(u) if u else {"symbol": sym, "date": date, "missing": True}, open(fp, "w"))
    return u
def run_units(jobs):
    rows = []
    with ThreadPoolExecutor(THREADS) as ex:
        futs = {ex.submit(get_unit, s, d): (s, d) for s, d in jobs}
        for i, f in enumerate(as_completed(futs), 1):
            try:
                u = f.result()
                if u and not u.get("missing"):
                    rows.append(u)
            except Exception as e:
                print("  !", futs[f], e)
            if i % 100 == 0:
                print(f"   {i}/{len(jobs)}")
    return pd.DataFrame(rows)


## The 96-date x 8-pair panel

In [ ]:
# ============================================================================
# CELL V2 — THE 96-DATE x 8-PAIR SPOT PANEL (Tables "panel" and "assets" of Paper 1)
# ============================================================================
U = run_units([(s, d) for s in PAIRS for d in PANEL_D])
U = U[U["coverage"] >= 0.8]
print(f"units: {len(U)} of {len(PAIRS) * len(PANEL_D)}")
mech = 1 - 2 / (EMA_W / BAR + 1)
panel = {c: {"Q": date_mean_ci(U, f"{c}_Q"), "S": date_mean_ci(U, f"{c}_S"), "J11": date_mean_ci(U, f"{c}_J11")}
         for c in ["forward", "observed", "raw_return", "observed_q30", "placebo"]}
assets = U.groupby("symbol")[["forward_Q", "observed_Q", "observed_q30_Q", "observed_S", "placebo_Q"]].mean().to_dict("index")
rows = [{"construction": c, "Q": round(v["Q"]["mean"], 7), "Q_CI": np.round(v["Q"]["ci"], 7).tolist(), "S": round(v["S"]["mean"], 5),
         "J11": round(v["J11"]["mean"], 5), "share Q>0": round(v["Q"]["share_units_pos"], 3)} for c, v in panel.items()]
print(pd.DataFrame(rows).to_string(index=False)); print("mechanical persistence 29/31 =", round(mech, 5))
print(pd.DataFrame(assets).T.round(7).to_string())
record("V2_spot_panel", None, "real-data", {"panel": panel, "assets": assets, "n_units": len(U),
       "n_transitions": int(U["n_bars"].sum() - len(U)), "mechanical_J11": mech},
       note="Re-run of the earlier version's spot panel from raw 1-second klines; adds a placebo pipeline.")


## Two fee events (this cell zips and downloads the results)

In [ ]:
# ============================================================================
# CELL V3 — FEE-SCHEDULE TEST WITH TWO EVENTS
# BTCUSDT spot fees went to zero on 2022-07-08 and came back on 2023-03-22. Seven other pairs are controls.
# Y_it = a_i + g_t + b * (BTC_i x ZeroFee_t) + e, separately for each event and pooled; date-clustered SEs.
# Outcomes: log quote volume (a first stage: fees should move volume) and observed-flow Q.
# ============================================================================
fee_jobs, ev_of = [], {}
for ev, d0 in FEE_EVENTS.items():
    for k_ in range(-FEE_WIN, FEE_WIN):
        d = (pd.Timestamp(d0) + pd.Timedelta(days=k_)).strftime("%Y-%m-%d")
        ev_of[d] = (ev, k_ >= 0); fee_jobs += [(s, d) for s in PAIRS]
FU = run_units(fee_jobs); FU = FU[FU["coverage"] >= 0.8].copy()
FU["event"] = FU["date"].map(lambda d: ev_of[d][0]); FU["post"] = FU["date"].map(lambda d: ev_of[d][1])
FU["btc"] = (FU["symbol"] == "BTCUSDT").astype(float)
FU["zero_fee"] = np.where(FU["event"] == "zero_fee_on", FU["post"], ~FU["post"]).astype(float) * FU["btc"]
FU["log_qv"] = np.log(FU["quote_volume_musd"])
def twfe(D, y):
    D = D.dropna(subset=[y]); X = pd.get_dummies(D[["symbol", "date"]], drop_first=True).astype(float)
    X.insert(0, "treat", D["zero_fee"].to_numpy()); X.insert(0, "const", 1.0)
    b, *_ = np.linalg.lstsq(X.to_numpy(), D[y].to_numpy(), rcond=None); e = D[y].to_numpy() - X.to_numpy() @ b
    Xm = X.to_numpy(); XtXi = np.linalg.pinv(Xm.T @ Xm); meat = np.zeros_like(XtXi)
    for _, idx in D.reset_index(drop=True).groupby("date").groups.items():
        s_ = Xm[idx].T @ e[idx]; meat += np.outer(s_, s_)
    G = D["date"].nunique(); V = XtXi @ meat @ XtXi * G / max(G - 1, 1); se = np.sqrt(V[1, 1])
    t_ = b[1] / se; p_ = 2 * (1 - stats.t.cdf(abs(t_), G - 1))
    return {"beta": float(b[1]), "se": float(se), "ci": [float(b[1] - stats.t.ppf(.975, G - 1) * se), float(b[1] + stats.t.ppf(.975, G - 1) * se)],
            "p": float(p_), "n": len(D), "dates": int(G)}
fee = {}
for ev in list(FEE_EVENTS) + ["pooled"]:
    D = FU if ev == "pooled" else FU[FU["event"] == ev]
    fee[ev] = {"log_quote_volume": twfe(D, "log_qv"), "observed_Q": twfe(D, "observed_Q")}
    print(ev, {k: {kk: round(vv, 5) if isinstance(vv, float) else vv for kk, vv in v.items()} for k, v in fee[ev].items()})
record("V3_fee_events", None, "real-data", {"by_event": fee, "window_days": FEE_WIN, "events": FEE_EVENTS},
       note="b = effect of zero fees on BTCUSDT relative to 7 control pairs; date-clustered. A volume effect "
            "with no Q effect says fees move activity but not measured feedback.")
zp8 = os.path.join(BASE, "p8_results.zip")
pd.DataFrame(DL_LOG).to_csv(os.path.join(RESULTS, "p8_manifest_sha256.csv"), index=False)
with zipfile.ZipFile(zp8, "w", zipfile.ZIP_DEFLATED) as z:
    for fp in glob.glob(os.path.join(RESULTS, "V*.json")) + [os.path.join(RESULTS, "p8_manifest_sha256.csv")]:
        z.write(fp, os.path.relpath(fp, BASE))
print("zipped ->", zp8)
if IN_COLAB:
    from google.colab import files
    files.download(zp8)


Send me `p8_results.zip`.